# S4 — Reproducibility analysis

This notebook reproduces the statistical analyses reported in the manuscript **“Temporal Integrity of Digital Traces in Project Team Analytics: Risk of Information Leakage and Cross-Project Model Transferability.”**

Inputs: `../S3_data/S3_modeling_dataset.csv` and `../S3_data/S3_outcome_robustness.csv`. Extended archived outputs are supplied in S5.

The original computational environment is listed in `environment.txt`. Random seed: 42. The SQL lineage from TAWOS v1.1 to S3 is provided in S1.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, log_loss)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_BOOT = 2000
HERE = Path.cwd()
DATA_DIR = (HERE.parent / 'S3_data') if (HERE.parent / 'S3_data').exists() else Path('../S3_data')

print('Python environment')
print('NumPy', np.__version__)
print('pandas', pd.__version__)
print('SciPy', scipy.__version__)
print('scikit-learn', sklearn.__version__)
print('statsmodels', sm.__version__)

Python environment
NumPy 2.3.5
pandas 2.2.3
SciPy 1.17.0
scikit-learn 1.8.0
statsmodels 0.14.6


## 1. Load and verify S3

In [2]:
df = pd.read_csv(DATA_DIR / 'S3_modeling_dataset.csv', parse_dates=['Start_Date','End_Date','prediction_time_20pct'])
outcomes = pd.read_csv(DATA_DIR / 'S3_outcome_robustness.csv', parse_dates=['Start_Date','End_Date'])
print('S3 shape:', df.shape)
print('Projects:', df['Project_Key'].nunique())
print('Primary successes:', int(df['success_90_end24'].sum()), 'rate=', df['success_90_end24'].mean())
assert df.shape == (952, 50)
assert df['Project_Key'].nunique() == 15
assert int(df['success_90_end24'].sum()) == 156
assert int(df['stable_from_start_flag'].sum()) == 3
assert int(df['stable_after_20pct_flag'].sum()) == 21

S3 shape: (952, 50)
Projects: 15
Primary successes: 156 rate= 0.1638655462184874


## 2. Temporal-integrity sensitivity (S5.1)

In [3]:
def stability_counts(theta):
    start = ((df.created_after_start_share <= theta) &
             (df.resolved_before_start_share <= 0.05) &
             (df.assignee_changed_issue_share <= theta) &
             (df.sprint_changed_issue_share <= theta))
    h20 = ((df.created_after_20pct_share <= theta) &
           (df.resolved_before_start_share <= 0.05) &
           (df.assignee_changed_after_20pct_issue_share <= theta) &
           (df.sprint_changed_after_20pct_issue_share <= theta))
    return int(start.sum()), int(h20.sum())

stab = pd.DataFrame([(t,*stability_counts(t)) for t in [0.05,0.10,0.15,0.20]],
                    columns=['theta','stable_start_n','stable_20pct_n'])
stab['stable_start_share'] = stab.stable_start_n / len(df)
stab['stable_20pct_share'] = stab.stable_20pct_n / len(df)
stab

,theta,stable_start_n,stable_20pct_n,stable_start_share,stable_20pct_share
0,0.05,1,7,0.001050,0.007353
1,0.10,3,21,0.003151,0.022059
2,0.15,7,40,0.007353,0.042017
3,0.20,18,79,0.018908,0.082983


## 3. Model specifications and temporal holdout

In [4]:
M0 = ['planned_duration_days','eligible_sprint_order','avg_prior_task_count',
      'experienced_member_share','assigned_issue_share']
M1 = M0 + ['created_between_start_and_20pct_count',
           'assignee_changed_before_20pct_issue_count',
           'sprint_changed_before_20pct_issue_count']
M2 = M1 + ['issue_count','core_team_size','bug_count','high_priority_issue_count',
           'created_after_start_count','assignee_changed_issue_count','sprint_changed_issue_count']
FEATURES = {'M0':M0,'M1':M1,'M2':M2}

def temporal_split(data):
    train_idx, test_idx = [], []
    ordered = data.sort_values(['Project_Key','Start_Date','Sprint_ID'])
    for _, g in ordered.groupby('Project_Key', sort=False):
        idx = g.index.to_numpy()
        n_train = int(np.floor(0.8 * len(idx)))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    return data.loc[train_idx].copy(), data.loc[test_idx].copy()

train, test = temporal_split(df)
print(len(train), len(test), int(test.success_90_end24.sum()))
assert (len(train),len(test),int(test.success_90_end24.sum())) == (755,197,30)

755 197 30


In [5]:
def make_model(numeric_features, include_project=True, C=1.0):
    transformers = [('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                                      ('scale', StandardScaler())]), numeric_features)]
    cols = list(numeric_features)
    if include_project:
        transformers.append(('project', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                                                   ('onehot', OneHotEncoder(handle_unknown='ignore'))]), ['Project_Key']))
        cols += ['Project_Key']
    prep = ColumnTransformer(transformers)
    model = Pipeline([('prep', prep),
                      ('logit', LogisticRegression(penalty='l2', C=C, solver='liblinear',
                                                   max_iter=5000, random_state=SEED))])
    return model, cols

def calibration_intercept_slope(y, p):
    p = np.clip(np.asarray(p), 1e-15, 1-1e-15)
    lp = np.log(p/(1-p))
    # A constant-probability baseline has no identifiable calibration slope.
    # Return NaN for intercept/slope instead of failing; discrimination/probability
    # metrics remain well-defined and are the quantities reported for baselines.
    if np.nanstd(lp) < 1e-12:
        return np.nan, np.nan
    X = sm.add_constant(lp, has_constant='add')
    fit = sm.GLM(np.asarray(y), X, family=sm.families.Binomial()).fit()
    return float(fit.params[0]), float(fit.params[1])

def metric_row(y, p):
    ci, cs = calibration_intercept_slope(y,p)
    return dict(ROC_AUC=roc_auc_score(y,p), AP=average_precision_score(y,p),
                Brier=brier_score_loss(y,p), LogLoss=log_loss(y,p),
                CalIntercept=ci, CalSlope=cs, MeanProbability=float(np.mean(p)))

def temporal_predictions(data_train, data_test, outcome='success_90_end24', include_project=True, C=1.0):
    preds = {}
    for name, features in FEATURES.items():
        model, cols = make_model(features, include_project=include_project, C=C)
        model.fit(data_train[cols], data_train[outcome])
        preds[name] = model.predict_proba(data_test[cols])[:,1]
    return preds

preds = temporal_predictions(train,test)
main_metrics = pd.DataFrame({name: metric_row(test.success_90_end24, p) for name,p in preds.items()}).T
main_metrics

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,ROC_AUC,AP,Brier,LogLoss,CalIntercept,CalSlope,MeanProbability
M0,0.743713,0.665835,0.081687,0.342805,0.092690,0.844259,0.121408
M1,0.732335,0.644465,0.082087,0.353162,0.021181,0.763496,0.115776
M2,0.755689,0.651080,0.081629,0.344131,0.008355,0.764175,0.118767


In [6]:
# Exact control values reported in the manuscript
expected = {
'M0': (0.7437125748502994,0.6658352347472929,0.08168666453250632,0.3428050357449782,0.8442592967368194),
'M1': (0.7323353293413174,0.6444645654867758,0.08208704206135803,0.3531622738538781,0.7634957670259676),
'M2': (0.7556886227544911,0.6510803885914354,0.08162861843407897,0.3441310066755536,0.7641749632080757)}
for m, vals in expected.items():
    got = main_metrics.loc[m, ['ROC_AUC','AP','Brier','LogLoss','CalSlope']].to_numpy(float)
    assert np.allclose(got, vals, atol=1e-10), (m,got,vals)
print('Main-model control metrics reproduced exactly.')

Main-model control metrics reproduced exactly.


## 4. Paired stratified bootstrap

In [7]:
def paired_stratified_bootstrap(y, pred_dict, n_boot=N_BOOT, seed=SEED):
    y=np.asarray(y); pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)
    rng=np.random.default_rng(seed)
    rows=[]
    for b in range(n_boot):
        idx=np.concatenate([rng.choice(pos,len(pos),replace=True), rng.choice(neg,len(neg),replace=True)])
        rng.shuffle(idx)
        yy=y[idx]
        r={'iteration':b}
        for name,p in pred_dict.items():
            pp=np.asarray(p)[idx]
            r[f'{name}_ROC_AUC']=roc_auc_score(yy,pp)
            r[f'{name}_AP']=average_precision_score(yy,pp)
            r[f'{name}_Brier']=brier_score_loss(yy,pp)
            r[f'{name}_LogLoss']=log_loss(yy,pp)
        rows.append(r)
    return pd.DataFrame(rows)

boot = paired_stratified_bootstrap(test.success_90_end24, preds)
ci_rows=[]
for m in FEATURES:
    for metric in ['ROC_AUC','AP']:
        v=boot[f'{m}_{metric}']
        ci_rows.append([m,metric,*np.quantile(v,[.025,.975])])
pd.DataFrame(ci_rows,columns=['Model','Metric','CI_low','CI_high'])

,Model,Metric,CI_low,CI_high
0,M0,ROC_AUC,0.607380,0.865299
1,M0,AP,0.517195,0.810283
2,M1,ROC_AUC,0.596203,0.856487
3,M1,AP,0.498496,0.792619
4,M2,ROC_AUC,0.627939,0.870080
5,M2,AP,0.503225,0.793852


## 5. Project-context removal, baselines and C sensitivity (S5.4)

In [8]:
no_project_preds = temporal_predictions(train,test,include_project=False)
no_project = pd.DataFrame({name: metric_row(test.success_90_end24,p) for name,p in no_project_preds.items()}).T
no_project[['ROC_AUC','AP','Brier','LogLoss']]

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,ROC_AUC,AP,Brier,LogLoss
M0,0.706786,0.363827,0.118963,0.409836
M1,0.704790,0.437114,0.117432,0.413803
M2,0.698603,0.299651,0.120528,0.415555


In [9]:
# Baselines use training outcomes only
ytr=train.success_90_end24
yte=test.success_90_end24
global_p=np.repeat(ytr.mean(),len(test))
project_rates=((train.groupby('Project_Key').success_90_end24.agg(['sum','count'])
                .assign(p=lambda x:(x['sum']+1)/(x['count']+2)))['p'])
project_p=test.Project_Key.map(project_rates).to_numpy()
baselines=pd.DataFrame({'global_training_frequency':metric_row(yte,global_p),
                        'smoothed_project_history':metric_row(yte,project_p)}).T
baselines[['ROC_AUC','AP','Brier','LogLoss']]

,ROC_AUC,AP,Brier,LogLoss
global_training_frequency,0.500000,0.152284,0.129307,0.427437
smoothed_project_history,0.717864,0.637785,0.085413,0.328334


In [10]:
c_rows=[]
for C in [0.1,1,10]:
    pp=temporal_predictions(train,test,C=C)
    for m,p in pp.items():
        r=metric_row(yte,p); c_rows.append([C,m,r['ROC_AUC'],r['AP'],r['Brier'],r['LogLoss']])
pd.DataFrame(c_rows,columns=['C','Model','ROC_AUC','AP','Brier','LogLoss'])

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,C,Model,ROC_AUC,AP,Brier,LogLoss
0,0.1,M0,0.758882,0.671099,0.095624,0.345289
1,0.1,M1,0.745309,0.646393,0.095340,0.348465
2,0.1,M2,0.755888,0.640413,0.097297,0.347990
3,1.0,M0,0.743713,0.665835,0.081687,0.342805
4,1.0,M1,0.732335,0.644465,0.082087,0.353162
5,1.0,M2,0.755689,0.651080,0.081629,0.344131
6,10.0,M0,0.731737,0.663856,0.079827,0.381838
7,10.0,M1,0.719561,0.648908,0.079593,0.394596
8,10.0,M2,0.732735,0.642465,0.079134,0.385314


## 6. Alternative operational outcomes (S5.3)

In [11]:
label_cols=['success_80_end0','success_80_end24','success_90_end0','success_90_end24',
            'success_90_end48','success_95_end24','success_90_end24_no_preresolved']
labels=outcomes[['Project_ID','Sprint_ID']+label_cols].copy()
base=df.drop(columns=[c for c in label_cols if c in df.columns]).merge(labels,on=['Project_ID','Sprint_ID'],how='left',validate='one_to_one')
alt_rows=[]
for outcome in label_cols:
    tr,te=temporal_split(base)
    pp=temporal_predictions(tr,te,outcome=outcome)
    for m,p in pp.items():
        r=metric_row(te[outcome],p)
        alt_rows.append([outcome,m,int(base[outcome].sum()),float(base[outcome].mean()),
                         int(te[outcome].sum()),float(te[outcome].mean()),
                         r['ROC_AUC'],r['AP'],r['Brier'],r['LogLoss'],r['CalIntercept'],r['CalSlope'],r['MeanProbability']])
alt=pd.DataFrame(alt_rows,columns=['Outcome','Model','Positive_all','Share_all','Positive_test','Share_test',
                                  'ROC_AUC','AP','Brier','LogLoss','CalIntercept','CalSlope','MeanProbability'])
alt

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,Outcome,Model,Positive_all,Share_all,Positive_test,Share_test,ROC_AUC,AP,Brier,LogLoss,CalIntercept,CalSlope,MeanProbability
0,success_80_end0,M0,245,0.257353,39,0.197970,0.753165,0.630716,0.116750,0.413784,-0.166565,0.750144,0.184073
1,success_80_end0,M1,245,0.257353,39,0.197970,0.754463,0.622763,0.117448,0.420020,-0.189038,0.699294,0.178585
2,success_80_end0,M2,245,0.257353,39,0.197970,0.768744,0.635363,0.116445,0.423686,-0.262044,0.659985,0.184497
3,success_80_end24,M0,287,0.301471,44,0.223350,0.705882,0.618922,0.139773,0.494550,-0.424268,0.558945,0.223057
4,success_80_end24,M1,287,0.301471,44,0.223350,0.707071,0.614685,0.140495,0.501281,-0.431899,0.531135,0.218360
5,success_80_end24,M2,287,0.301471,44,0.223350,0.718063,0.587149,0.144436,0.525186,-0.521071,0.450903,0.222575
6,success_90_end0,M0,125,0.131303,22,0.111675,0.837403,0.780802,0.051401,0.221882,0.740158,1.199477,0.088498
7,success_90_end0,M1,125,0.131303,22,0.111675,0.837662,0.764648,0.050116,0.224237,0.568352,1.067039,0.086382
8,success_90_end0,M2,125,0.131303,22,0.111675,0.854286,0.759542,0.050524,0.218202,0.456345,1.058355,0.091148
9,success_90_end24,M0,156,0.163866,30,0.152284,0.743713,0.665835,0.081687,0.342805,0.092690,0.844259,0.121408


## 7. Leave-One-Project-Out (LOPO) and project decomposition (S5.2)

In [12]:
def leave_one_group_out(data, group_col, outcome='success_90_end24'):
    all_preds={m:np.full(len(data),np.nan) for m in FEATURES}
    for group in data[group_col].drop_duplicates():
        tr=data[data[group_col]!=group]; te=data[data[group_col]==group]
        for name,features in FEATURES.items():
            model,cols=make_model(features,include_project=False,C=1)
            model.fit(tr[cols],tr[outcome])
            all_preds[name][te.index]=model.predict_proba(te[cols])[:,1]
    return all_preds

lopo_preds=leave_one_group_out(df,'Project_Key')
lopo_metrics=pd.DataFrame({m:metric_row(df.success_90_end24,p) for m,p in lopo_preds.items()}).T
lopo_metrics

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,ROC_AUC,AP,Brier,LogLoss,CalIntercept,CalSlope,MeanProbability
M0,0.558087,0.180118,0.143602,0.459837,-1.127435,0.330576,0.195260
M1,0.560455,0.178479,0.148203,0.473290,-1.272539,0.228393,0.194778
M2,0.478957,0.156321,0.165214,0.537003,-1.772019,-0.085529,0.204417


In [13]:
def project_auc_table(data,pred):
    tmp=data[['Project_Key','success_90_end24']].copy(); tmp['pred']=pred
    rows=[]
    for pk,g in tmp.groupby('Project_Key'):
        y=g.success_90_end24.to_numpy(); npos=int(y.sum()); nneg=len(y)-npos
        auc=np.nan if npos==0 or nneg==0 else roc_auc_score(y,g.pred)
        rows.append([pk,len(g),npos,nneg,auc,npos*nneg])
    return pd.DataFrame(rows,columns=['Project_Key','N','Positive','Negative','ROC_AUC','PairWeight'])

def cross_project_pair_auc(data,pred):
    y=data.success_90_end24.to_numpy(); groups=data.Project_Key.to_numpy(); pred=np.asarray(pred)
    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0); score=0.0; denom=0
    for i in pos:
        js=neg[groups[neg]!=groups[i]]
        denom += len(js)
        score += np.sum(pred[i]>pred[js]) + 0.5*np.sum(pred[i]==pred[js])
    return score/denom

proj_tables={}; summary=[]
for m,p in lopo_preds.items():
    t=project_auc_table(df,p); proj_tables[m]=t
    valid=t.dropna(subset=['ROC_AUC'])
    within=np.average(valid.ROC_AUC,weights=valid.PairWeight)
    cross=cross_project_pair_auc(df,p)
    summary.append([m,roc_auc_score(df.success_90_end24,p),valid.ROC_AUC.mean(),valid.ROC_AUC.median(),within,cross,int((valid.ROC_AUC>.5).sum()),len(valid)])
pd.DataFrame(summary,columns=['Model','Pooled','MeanProject','MedianProject','WithinPairWeighted','CrossProjectPairs','ProjectsAbove0.5','ValidProjects'])

,Model,Pooled,MeanProject,MedianProject,WithinPairWeighted,CrossProjectPairs,ProjectsAbove0.5,ValidProjects
0,M0,0.558087,0.467069,0.529412,0.663193,0.547540,7,13
1,M1,0.560455,0.492894,0.588235,0.653744,0.551093,7,13
2,M2,0.478957,0.559074,0.595238,0.678206,0.458964,10,13


## 8. Project-cluster bootstrap and LOPO jackknife (S5.5)

In [14]:
def cluster_bootstrap_test(y,p0,p2,groups,n_boot=N_BOOT,seed=SEED):
    y=np.asarray(y); p0=np.asarray(p0); p2=np.asarray(p2); groups=np.asarray(groups)
    units=np.unique(groups); rng=np.random.default_rng(seed); vals=[]
    for _ in range(n_boot):
        sampled=rng.choice(units,size=len(units),replace=True)
        idx=np.concatenate([np.flatnonzero(groups==g) for g in sampled])
        yy=y[idx]
        if np.unique(yy).size<2: continue
        vals.append([roc_auc_score(yy,p2[idx])-roc_auc_score(yy,p0[idx]),
                     average_precision_score(yy,p2[idx])-average_precision_score(yy,p0[idx]),
                     brier_score_loss(yy,p2[idx])-brier_score_loss(yy,p0[idx]),
                     log_loss(yy,p2[idx])-log_loss(yy,p0[idx])])
    a=np.asarray(vals); names=['ROC_AUC','AP','Brier','LogLoss']
    point=np.array([roc_auc_score(y,p2)-roc_auc_score(y,p0),
                    average_precision_score(y,p2)-average_precision_score(y,p0),
                    brier_score_loss(y,p2)-brier_score_loss(y,p0),
                    log_loss(y,p2)-log_loss(y,p0)])
    return pd.DataFrame({'Metric':names,'PointDifference_M2_minus_M0':point,
                         'CI_low':np.quantile(a,.025,axis=0),'CI_high':np.quantile(a,.975,axis=0)})

cluster_bootstrap_test(test.success_90_end24,preds['M0'],preds['M2'],test.Project_Key)

,Metric,PointDifference_M2_minus_M0,CI_low,CI_high
0,ROC_AUC,0.011976,-0.029108,0.060602
1,AP,-0.014755,-0.100052,0.027602
2,Brier,-0.000058,-0.003135,0.002466
3,LogLoss,0.001326,-0.020130,0.015853


In [15]:
def lopo_decomposition(data,pred):
    pooled=roc_auc_score(data.success_90_end24,pred)
    t=project_auc_table(data,pred).dropna(subset=['ROC_AUC'])
    within=np.average(t.ROC_AUC,weights=t.PairWeight)
    cross=cross_project_pair_auc(data,pred)
    ci,cs=calibration_intercept_slope(data.success_90_end24,pred)
    return np.array([pooled,within,cross,cs])

# delete-one-project jackknife; normal approximation from jackknife SE
units=df.Project_Key.drop_duplicates().to_list(); estimates=[]
for omit in units:
    keep=df.Project_Key!=omit
    estimates.append(lopo_decomposition(df.loc[keep].reset_index(drop=True), np.asarray(lopo_preds['M2'])[keep.to_numpy()]))
estimates=np.asarray(estimates); full=lopo_decomposition(df,lopo_preds['M2'])
mean_j=estimates.mean(axis=0); se=np.sqrt((len(units)-1)/len(units)*np.sum((estimates-mean_j)**2,axis=0))
jk=pd.DataFrame({'Metric':['pooled_ROC','within_ROC','cross_ROC','calibration_slope'],
                 'Estimate':full,'CI_low':full-1.96*se,'CI_high':full+1.96*se})
jk

,Metric,Estimate,CI_low,CI_high
0,pooled_ROC,0.478957,0.246249,0.711665
1,within_ROC,0.678206,0.565641,0.790770
2,cross_ROC,0.458964,0.212120,0.705808
3,calibration_slope,-0.085529,-0.872953,0.701896


## 9. Leave-One-Repository-Out (S5.6)

In [16]:
loro_preds=leave_one_group_out(df,'repository_name')
loro=pd.DataFrame({m:metric_row(df.success_90_end24,p) for m,p in loro_preds.items()}).T
loro

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,ROC_AUC,AP,Brier,LogLoss,CalIntercept,CalSlope,MeanProbability
M0,0.559496,0.197335,0.143492,0.460044,-1.112014,0.343043,0.197185
M1,0.561107,0.185374,0.148568,0.475861,-1.273764,0.225440,0.194606
M2,0.479634,0.159498,0.167282,0.545216,-1.730765,-0.059654,0.205028


## Reproducibility note

The deterministic train/test metrics above are checked against exact archived values. Bootstrap and jackknife procedures are stochastic/finite-sample uncertainty estimators; S5 contains the archived numerical outputs used in the manuscript. The code uses the stated seed and 2,000 repetitions and is intended to reproduce those procedures under the pinned environment listed in `environment.txt`.